# SQL聚合 — COUNT / SUM / AVG / GROUP BY / HAVING（习题）

In [1]:
import duckdb

%load_ext sql
%sql duckdb://

Connecting to 'duckdb://'

Easy

1. 算出整张表的:总订单数、总销售额、平均客单价(三个值在一个查询里)。

In [10]:
duckdb.sql("""
    SELECT 
        COUNT(*) AS order_count,
        SUM(total) AS revenue,
        ROUND(AVG(total), 2) AS revenue_per_order
    FROM '../data/sales.csv' 
""")

┌─────────────┬─────────┬───────────────────┐
│ order_count │ revenue │ revenue_per_order │
│    int64    │ int128  │      double       │
├─────────────┼─────────┼───────────────────┤
│         500 │ 1267716 │           2535.43 │
└─────────────┴─────────┴───────────────────┘

2. 算出数据集里有多少个不重复的客户(customer_id)。提示:COUNT(DISTINCT ...)。

In [ ]:
duckdb.sql("""
    SELECT 
        COUNT(DISTINCT customer_id) AS customer_count_distinct
    FROM '../data/sales.csv' 
""")

┌─────────────────────────┐
│ customer_count_distinct │
│          int64          │
├─────────────────────────┤
│                       8 │
└─────────────────────────┘

3. 按 country 分组,算出每个国家的订单数,按订单数降序排列。

In [14]:
duckdb.sql("""
    SELECT 
        country,
        COUNT(*) AS order_count
    FROM '../data/sales.csv'
    GROUP BY country
    ORDER BY order_count DESC
""")


┌─────────┬─────────────┐
│ country │ order_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ UK      │         197 │
│ US      │         125 │
│ France  │          80 │
│ Germany │          66 │
│ China   │          32 │
└─────────┴─────────────┘

4. 按 category 分组,算出每个品类的总销售额(SUM(total)),按销售额降序。

In [ ]:
duckdb.sql("""
    SELECT 
        category, 
        SUM(total) AS revenue
    FROM '../data/sales.csv' 
    GROUP BY category
    ORDER BY revenue DESC
""")

┌───────────┬─────────┐
│ category  │ revenue │
│  varchar  │ int128  │
├───────────┼─────────┤
│ Accessory │  441266 │
│ Computer  │  363933 │
│ Mobile    │  257831 │
│ Audio     │  204686 │
└───────────┴─────────┘

Medium

5. 按 product 分组,算出每个产品的:订单数、总销售额、平均客单价(ROUND 保留 2 位小数),按总销售额降序。

In [12]:
duckdb.sql("""
    SELECT 
        product, 
        COUNT(*) AS order_count,
        SUM(total) AS revenue,
        ROUND(AVG(total), 2) AS revenue_per_order
    FROM '../data/sales.csv' 
    GROUP BY product
    ORDER BY revenue DESC
""")

┌────────────┬─────────────┬─────────┬───────────────────┐
│  product   │ order_count │ revenue │ revenue_per_order │
│  varchar   │    int64    │ int128  │      double       │
├────────────┼─────────────┼─────────┼───────────────────┤
│ Phone      │          93 │  257831 │           2772.38 │
│ Keyboard   │          99 │  225111 │           2273.85 │
│ Mouse      │          81 │  216155 │           2668.58 │
│ Headphones │          67 │  204686 │           3055.01 │
│ Laptop     │          87 │  200939 │           2309.64 │
│ Monitor    │          73 │  162994 │           2232.79 │
└────────────┴─────────────┴─────────┴───────────────────┘

6. 按 customer_id 分组,找出每个客户的总消费额,按消费额降序,只显示前 5 名(大客户榜)。

💡 这就是 Day 2 第 7 题"用户消费排序"的 SQL 版,对比一下两种写法。

In [15]:
duckdb.sql("""
    SELECT 
        customer_id, 
        SUM(total) AS revenue
    FROM '../data/sales.csv' 
    GROUP BY customer_id
    ORDER BY revenue DESC
    LIMIT 5
""")

┌─────────────┬─────────┐
│ customer_id │ revenue │
│   varchar   │ int128  │
├─────────────┼─────────┤
│ C006        │  198806 │
│ C001        │  184001 │
│ C008        │  164484 │
│ C003        │  160212 │
│ C004        │  144095 │
└─────────────┴─────────┘

7. 用 HAVING 找出订单数超过 60 的国家,显示国家名和订单数。

In [ ]:
duckdb.sql("""
    SELECT 
        country, 
        COUNT(*) AS order_count
    FROM '../data/sales.csv' 
    GROUP BY country
    HAVING order_count > 60
""")

┌─────────┬─────────────┐
│ country │ order_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ Germany │          66 │
│ UK      │         197 │
│ US      │         125 │
│ France  │          80 │
└─────────┴─────────────┘

参考答案

In [20]:
duckdb.sql("""
    SELECT 
        country, 
        COUNT(*) AS order_count
    FROM '../data/sales.csv' 
    GROUP BY country
    HAVING COUNT(*) > 60
""")

┌─────────┬─────────────┐
│ country │ order_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ Germany │          66 │
│ UK      │         197 │
│ France  │          80 │
│ US      │         125 │
└─────────┴─────────────┘

HAVING 在 SELECT 之前执行。也就是说,执行 HAVING 时,别名 order_count 还不存在(别名是 SELECT 阶段才创建的)。

严格标准的 SQL 里,HAVING order_count > 60 应该报错。DuckDB 和 MySQL 比较"贴心"地允许了这种写法,但 PostgreSQL、SQL Server 等会直接报错。

✅ 标准写法:HAVING 里要重写聚合函数,不能用别名:

8. WHERE + HAVING 组合:先只保留 total > 1000 的订单,再按 category 分组,找出这些大额订单的总金额超过 50000 的品类,显示品类、订单数、总金额。

In [17]:
duckdb.sql("""
    SELECT 
        category, 
        COUNT(*) AS order_count,
        SUM(total) AS revenue
    FROM '../data/sales.csv' 
    WHERE total > 1000
    GROUP BY category
    HAVING revenue > 50000
""")

┌───────────┬─────────────┬─────────┐
│ category  │ order_count │ revenue │
│  varchar  │    int64    │ int128  │
├───────────┼─────────────┼─────────┤
│ Computer  │          99 │  337075 │
│ Audio     │          46 │  193840 │
│ Mobile    │          69 │  246386 │
│ Accessory │         112 │  404723 │
└───────────┴─────────────┴─────────┘

参考答案

In [22]:
duckdb.sql("""
    SELECT 
        category, 
        COUNT(*) AS order_count,
        SUM(total) AS revenue
    FROM '../data/sales.csv' 
    WHERE total > 1000
    GROUP BY category
    HAVING SUM(total) > 50000
    ORDER BY revenue DESC
""")

┌───────────┬─────────────┬─────────┐
│ category  │ order_count │ revenue │
│  varchar  │    int64    │ int128  │
├───────────┼─────────────┼─────────┤
│ Accessory │         112 │  404723 │
│ Computer  │          99 │  337075 │
│ Mobile    │          69 │  246386 │
│ Audio     │          46 │  193840 │
└───────────┴─────────────┴─────────┘

Hard

9. 二维分组(Day 3 第 10 题的 SQL 版):按 country + category 两个维度分组,算出每个组合的总销售额,按 country 升序、销售额降序排列。

💡 对比一下:当年 Day 3 你写了十几行 Python 嵌套 dict,现在 SQL 几行搞定。把这个对比写进笔记。

In [18]:
duckdb.sql("""
    SELECT 
        country,
        category, 
        SUM(total) AS revenue
    FROM '../data/sales.csv' 
    GROUP BY country, category
    ORDER BY country, revenue DESC
""")

┌─────────┬───────────┬─────────┐
│ country │ category  │ revenue │
│ varchar │  varchar  │ int128  │
├─────────┼───────────┼─────────┤
│ China   │ Computer  │   32159 │
│ China   │ Accessory │   21271 │
│ China   │ Mobile    │   18480 │
│ China   │ Audio     │    8191 │
│ France  │ Audio     │   60245 │
│ France  │ Accessory │   54935 │
│ France  │ Computer  │   51431 │
│ France  │ Mobile    │   41554 │
│ Germany │ Computer  │   51036 │
│ Germany │ Accessory │   45042 │
│ Germany │ Mobile    │   28967 │
│ Germany │ Audio     │   18975 │
│ UK      │ Accessory │  164595 │
│ UK      │ Computer  │  158700 │
│ UK      │ Mobile    │  137788 │
│ UK      │ Audio     │   73734 │
│ US      │ Accessory │  155423 │
│ US      │ Computer  │   70607 │
│ US      │ Audio     │   43541 │
│ US      │ Mobile    │   31042 │
└─────────┴───────────┴─────────┘
  20 rows             3 columns

10. 综合分析题:回答这个业务问题——"哪些客户是高价值客户?"

定义:高价值客户 = 总消费额 ≥ 30000 且 下单次数 ≥ 8 次。

要求显示:customer_id、下单次数(order_count)、总消费额(total_spent)、平均客单价(avg_order,保留 2 位小数),按总消费额降序排列。

💡 这道题非常接近真实工作中的"用户分层分析",WHERE/GROUP BY/HAVING/ORDER BY 全都要用上。

In [19]:
duckdb.sql("""
    SELECT 
        customer_id,
        COUNT(*) AS order_count,
        SUM(total) AS total_spent,
        ROUND(AVG(total), 2) AS avg_order
    FROM '../data/sales.csv' 
    GROUP BY customer_id
    HAVING total_spent >= 30000 AND order_count >= 8
    ORDER BY total_spent DESC
""")

┌─────────────┬─────────────┬─────────────┬───────────┐
│ customer_id │ order_count │ total_spent │ avg_order │
│   varchar   │    int64    │   int128    │  double   │
├─────────────┼─────────────┼─────────────┼───────────┤
│ C006        │          62 │      198806 │   3206.55 │
│ C001        │          71 │      184001 │   2591.56 │
│ C008        │          72 │      164484 │    2284.5 │
│ C003        │          64 │      160212 │   2503.31 │
│ C004        │          76 │      144095 │   1895.99 │
│ C007        │          58 │      143917 │   2481.33 │
│ C002        │          46 │      140967 │    3064.5 │
│ C005        │          51 │      131234 │   2573.22 │
└─────────────┴─────────────┴─────────────┴───────────┘

参考答案

In [23]:
duckdb.sql("""
    SELECT 
        customer_id,
        COUNT(*) AS order_count,
        SUM(total) AS total_spent,
        ROUND(AVG(total), 2) AS avg_order
    FROM '../data/sales.csv' 
    GROUP BY customer_id
    HAVING SUM(total) >= 30000 AND COUNT(*) >= 8
    ORDER BY total_spent DESC
""")

┌─────────────┬─────────────┬─────────────┬───────────┐
│ customer_id │ order_count │ total_spent │ avg_order │
│   varchar   │    int64    │   int128    │  double   │
├─────────────┼─────────────┼─────────────┼───────────┤
│ C006        │          62 │      198806 │   3206.55 │
│ C001        │          71 │      184001 │   2591.56 │
│ C008        │          72 │      164484 │    2284.5 │
│ C003        │          64 │      160212 │   2503.31 │
│ C004        │          76 │      144095 │   1895.99 │
│ C007        │          58 │      143917 │   2481.33 │
│ C002        │          46 │      140967 │    3064.5 │
│ C005        │          51 │      131234 │   2573.22 │
└─────────────┴─────────────┴─────────────┴───────────┘

附加题：下面这条 SQL 错在哪里?

In [24]:
# SELECT country, product, SUM(total) AS revenue
# FROM 'data/sales.csv'
# GROUP BY country

回答：product 既不在 GROUP BY 中，也没有被聚合函数包裹。

我的弱点清单

9. HAVING 里写聚合函数本身,不要用 SELECT 的别名 —— DuckDB 容忍,但 PostgreSQL 等会报错(高频面试陷阱)

10. 别名可用性规则:ORDER BY 能用别名,WHERE/GROUP BY/HAVING 不能用(取决于执行顺序)

11. SELECT 最后一列后面不要加逗号 —— 养成习惯,换数据库不踩坑